# Open Design on Jute — Asset Library Design Spec

**Date:** 2026-06-01 · **Status:** Draft for review · **Authors:** brain (Claude) + Kevin
**Depends on:** `2026-05-31-open-design-jute-host-shell-design.ipynb` (host-shell, approved & M1 merged)

> Make Open Design's **123 skills + 149 design systems** available to the single bundled
> `open-design` SPUR skill **without skill-explosion** — by treating them as an on-disk
> **library** (data, not skills) consulted on demand via progressive disclosure.

This design was cross-checked against (a) Open Design's *actual* skill-management code and
(b) industry best practice (Anthropic Agent Skills, MCP resources/tools). Both are folded in.


## Context — what M1 shipped vs. what this adds

**M1 (merged):** the `open-design` brain skill = the *loop* (discovery → direction → plan →
artifact → critique) + the **5 directions** + the 5-dimensional / anti-AI-slop critique. It
proves the notebook-as-visualization-layer mechanism end to end.

**This milestone:** the *asset library* M1 deliberately deferred —

| Asset | Count | Reality (measured) |
|---|---|---|
| Artifact-mode skills | **~30** | saas-landing, dashboard, mobile-app, invoice, finance-report, deck… (structure + seed `template.html` + `checklist.md`) |
| Theme variants | **51 `html-ppt-*`** + 10 `*-template` + `web-prototype-taste-*` | NOT distinct modes — themes of a parent mode |
| Design systems | **148** (`DESIGN.md`, **1.8 MB total**) | branded palette/type tokens (Linear, Stripe, …) |

The hard constraint: **these cannot become 272 bundled SPUR skills.** Quantified below.


## Verified finding A — how Open Design actually manages skills

Read from `apps/daemon/src/{skills,design-systems,library-install,frontmatter}.ts`:

1. **Runtime filesystem scan, not a compiled registry.** `listSkills(root)` reads
   `skills/*/SKILL.md`, parses YAML frontmatter, derives `SkillInfo`. Source comment:
   *"No watching in this MVP — re-scans on every `GET /api/skills`, fine for dozens of skills."*
2. **Frontmatter is the index** — `name`, `description`, `triggers[]`, and an `od:` block
   (`mode`, `surface`, `platform`, `scenario`, `preview.type`, `design_system.requires`,
   `defaultFor`, `featured`, `fidelity`).
3. **Progressive disclosure is explicit.** `withSkillRootPreamble()` injects the skill-root
   path + a *"Known side files: …"* list; the agent `Read`s `template.html` / `layouts.md` /
   `checklist.md` **on demand**. Side files are never inlined.
4. **Layered + user-extensible.** Bundled `skills/` + user-installed `~/.open-design/skills/`
   (`library-install.ts` clones from GitHub or copies local) — a marketplace pattern.
5. **Theme variants are derived children.** The 51 `html-ppt-*` are `aggregatesExamples`
   parent/child derivations, not top-level skills.
6. **Design systems**: scan `design-systems/*/DESIGN.md`; title from H1, category from a
   `> Category:` line, swatches extracted.


## Verified finding B — industry best practice

**Anthropic Agent Skills — 3-layer progressive disclosure** (OD and SPUR skills already implement it):

| Layer | Loaded | Cost |
|---|---|---|
| **Discovery** | `name` + `description` only, always | ~80 tokens / skill |
| **Activation** | full `SKILL.md` body, when relevant | ~275–8k (median ~2k) |
| **Execution** | bundled scripts / templates / refs, on demand | only when reached |

Rules: `SKILL.md` **< 500 lines** (split into reference files; keep mutually-exclusive paths
separate); `name` ≤64 chars `[a-z0-9-]`; `description` ≤1024, must state **what + when**.

**MCP resources vs tools:** Tools = model-invoked **actions with side effects**. Resources =
**read-only data**, URI-addressed, **app-fetched** + host-injected. For datasets **> ~50 KB or
exceeding context**, expose **resource URIs**, not tool responses, so the host controls loading.


## Locked decisions

| # | Decision | Choice | Backed by |
|---|----------|--------|-----------|
| 1 | Skill explosion? | **One** bundled `open-design` skill; library is **data, not skills** | OD scans (not compiles); 272×80 ≈ 22k discovery tokens otherwise |
| 2 | Where assets live | On-disk **vendored library**, read on demand — **never `include_str!`** | Anthropic Execution layer; OD filesystem scan |
| 3 | Access primitive | **`Read` by default** (portable, = OD); MCP **Resources** as idiomatic option; a **single `open_design_search` tool** for ranking only | MCP: read-only data → Resources/Read, not tools |
| 4 | Index timing | Index consulted **on demand at selection (Execution layer)**, never at Discovery | progressive-disclosure token budget |
| 5 | Theme variants | 51 `html-ppt-*` fold under their parent mode as **themes** | OD `aggregatesExamples` parent/child |
| 6 | Extensibility | Bundled defaults + user adds in `.spur/open-design/` | OD `library-install.ts` + SPUR `.spur/skills` precedence |


## Three-layer asset model

```mermaid
flowchart TB
  brief([Brief]) --> sel{open-design loop · selection}
  sel -->|mode by trigger/scenario| modes["Artifact modes (~30)\nstructure + seed template + checklist"]
  sel -->|branded?| ds["Design systems (148)\npalette + type tokens"]
  sel -->|no brand| dir["5 directions (M1)\nquick palette pick"]
  modes --> themes["Theme variants (51 html-ppt + 10 template)\nfolded UNDER a mode"]
  modes --> art["Artifact cell · text/html"]
  ds --> art
  dir --> art
  themes --> art
  classDef have fill:#e8f5e9,stroke:#43a047;
  classDef lib fill:#e3f2fd,stroke:#1e88e5;
  class dir have;
  class modes,ds,themes lib;
```

🟢 already in M1 · 🔵 the library this spec adds. Mode and design system **compose** (mode = structure, design system/direction = palette).


## Storage — vendored library, read on demand

```
crates/spur-notebook/assets/open-design-library/      (tracked, shipped, NOT include_str!)
├── skills/<id>/
│     ├── SKILL.md            # mode instructions (frontmatter: mode, scenario, platform, triggers, theme_of?)
│     ├── template.html       # seed
│     ├── layouts.md          # paste-ready section skeletons
│     └── checklist.md        # P0/P1/P2 self-check
├── design-systems/<id>/DESIGN.md     # all 148 (1.8 MB, vendor as-is)
└── index.json                        # generated, compact (Discovery/Execution metadata)
```

- A **build script** generates `index.json` and **strips the 43 MB of example media** at vendor
  time (instructions + templates are small; the bulk is screenshots).
- Installed/extracted to a runtime dir on first use (same pattern as `.spur/skills/`), or read
  in place. User additions live in `.spur/open-design/{skills,design-systems}/` and take
  precedence (mirrors OD `~/.open-design/` + SPUR override dirs).


## Access design (refined — Read/Resources, not custom tools)

```mermaid
flowchart LR
  need["open-design loop needs a mode / design system"] --> q{access path}
  q -->|default · portable| rd["Agent Read\nindex.json + package side files\n(= exactly what OD does)"]
  q -->|idiomatic · host-managed| res["MCP Resources\nopendesign://skills/{id}\nopendesign://design-systems/{id}"]
  q -->|the ONE action| tool["open_design_search(brief|mode|scenario|brand)\n→ ranked [{id, kind, summary, palette}]"]
  tool --> rd
  tool --> res
  classDef rec fill:#e8f5e9,stroke:#43a047;
  class rd,tool rec;
```

- **Default = `Read`** — most portable, works on every agent client, matches OD 1:1.
- **MCP Resources** — idiomatic for read-only data + host-managed context loading; adopt only
  if target clients consume resources well. Good for the >50 KB packages.
- **`open_design_search`** is the only thing modeled as a **tool** — it computes a ranking/
  decision (a real action), unlike fetching a doc.


## Selection flow — two steps added to the `open-design` loop

```mermaid
sequenceDiagram
  participant U as Designer
  participant A as Brain (open-design skill)
  participant L as Library (Read / Resource / search)
  participant N as Notebook
  U->>A: brief
  A->>N: insert discovery cell; read answers
  A->>L: open_design_search(mode, scenario)
  L-->>A: ranked mode candidates
  A->>L: get skill <id> (SKILL.md + template + checklist)
  Note over A: pre-flight read (Execution layer)
  A->>L: branded? search design-systems ; get DESIGN.md
  Note over A: else use the 5 directions (M1)
  A->>N: write artifact cell (seed template bound to palette) → text/html
  A->>A: critique via skill checklist.md + 5-dim (M1)
  A->>N: write revised artifact
```


## Token-budget check (why this stays in-bounds)

| Approach | Discovery cost | Verdict |
|---|---|---|
| 272 bundled skills | 272 × ~80 ≈ **22k tokens always loaded** | ❌ violates progressive disclosure |
| **This design** | 1 skill (`open-design`) ≈ ~80 tokens at Discovery | ✅ |
| Index at selection | skills ~30 × ~40 ≈ 1.2k; design-systems ~148 × ~30 ≈ 4.5k — **read on demand only** | ✅ Execution layer |
| One package | one `SKILL.md` + `template.html` read when chosen | ✅ Execution layer |

The brain never holds the whole library — only the one skill at Discovery, the index at the
moment of selection, and the single chosen package.


## UI/UX mockup — library picker (rendered as a text/html cell output)

Illustrative: mode × design-system selection surfaced from the index. Static-styled;
the search box is a progressive-enhancement stub.


In [1]:
# Library picker mockup — mode x design-system selection from the index

## Milestones

- **M2 — Deck mode + deck themes.** Route `kind: deck` into Jute deck mode; bring the 51
  `html-ppt-*` as deck themes (parent/child). Natural pairing with the existing deck UI.
- **M3.5 — Design-system library (recommended next).** Vendor all 148 `DESIGN.md` (1.8 MB) +
  generate the design-systems index; wire the loop's Direction step to search/get them.
  High value, near-zero risk.
- **M4 — Mode-skill library + access surface.** Vendor the ~30 mode packages (media stripped);
  generate the skills index; add `open_design_search` (+ optional MCP Resources); wire the
  loop's mode-selection step; critique via each skill's `checklist.md`.
- **M4+ — User marketplace.** GitHub/local install into `.spur/open-design/` (port
  `library-install.ts` semantics behind SPUR's path/safety rules).


## Open decisions to settle before a plan

1. **Access surface for M4:** ship `Read`-only first, or `Read` + `open_design_search` tool, or
   add MCP Resources? (Lean: `Read` + `open_design_search`; Resources later.)
2. **Vendoring scope:** all 148 design systems + ~30 modes day one, or a curated subset first?
3. **Media handling:** strip example media entirely, or keep thumbnails out-of-band?
4. **Index generation:** prebuilt `index.json` (committed) vs. scan-on-demand like OD
   (simpler, "fine for dozens/hundreds")?
5. **Theme model:** represent `html-ppt-*` as `theme_of:` frontmatter on child packages, or as
   a `themes/` subdir inside the `deck` mode package?

**Next:** on approval, turn **M3.5 (design-system library)** into the next `submit_plan`.


## Sources

- Anthropic — *Equipping agents for the real world with Agent Skills* (engineering blog)
- Claude Docs — *Skill authoring best practices* (SKILL.md < 500 lines; name/description rules)
- SwirlAI — *Agent Skills: Progressive Disclosure as a System Design Pattern* (3-layer cost model)
- Microsoft Community — *MCP Demystified: Tools vs Resources vs Prompts*
- *MCP Resources explained (and how they differ from Tools)* — >50 KB → resource URIs
- Open Design source: `apps/daemon/src/{skills,design-systems,library-install,frontmatter}.ts`
